# 05 — Extracción de Características
## Pipeline CAD-TBX11K | Diagnóstico Asistido por Computador en Tuberculosis

Este notebook construye la **matriz de características** que sirve como entrada a
los modelos de machine learning. Cada radiografía se convierte en un vector numérico
de **1860 características** calculadas sobre el campo pulmonar segmentado.

### Grupos de características del modelo

| Grupo   | Qué captura                                    | Features |
|---------|------------------------------------------------|:--------:|
| Textura | intensidad, GLCM, LBP y Gabor dentro del pulmón |    96    |
| HOG     | gradientes orientados sobre la vista lung (mediana) |  1764    |
| **Total** |                                              | **1860** |

shape, context y quality existen en src/features.py para análisis de
sesgo, pero no se incluyen en la matriz principal del modelo.

### Pipeline por imagen

`
imagen → preprocess_bundle() → recorte torácico + máscara
                              → vista lung (exterior = mediana pulmonar)
                              → textura (GLCM, LBP, Gabor)
                              → HOG sobre la vista lung
                              → fila del CSV
`


## 0. Configuración del entorno

In [ ]:
# Se configura el path del proyecto y se importan las funciones del framework.

import sys, os, warnings
warnings.filterwarnings('ignore')

_CWD = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_CWD, '..')) if os.path.basename(_CWD) == 'notebooks' else _CWD
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

from src.data import download_dataset, get_data_root, build_base_dataframe, LABEL_MAP

# Configuración central del pipeline: definida una sola vez en src/config.py
from src.config import N_MODELO
from src.features import (
    verify_extractors,        # Verifica el número correcto de features por extractor
    extract_all_features,     # Extracción masiva sobre el DataFrame base
    get_feature_groups,       # Clasifica columnas por grupo de extractor
    print_feature_summary,    # Resumen de integridad del DataFrame de features
    REGIONS, GLCM_PROPS,
    LBP_P, LBP_R, HOG_PIXELS_PER_CELL,
)
from src.visualization import (
    set_publication_style,
    plot_glcm_distributions,
    plot_lbp_histograms,
    plot_feature_summary_table,
)


from pathlib import Path
OUTPUT_CSV = Path(PROJECT_ROOT) / 'data' / 'features_tbx11k.csv'

set_publication_style()
print('Entorno configurado correctamente.')

## 1. Carga del DataFrame base

In [ ]:
# Se carga el DataFrame canonico desde data/df_model.csv (8.256 imagenes).
# Si no existe, se genera desde df_base.csv con la lista canonica de duplicados.
from pathlib import Path
CSV_PATH = Path(PROJECT_ROOT) / 'data' / 'df_model.csv'

if not CSV_PATH.exists():
    import subprocess
    subprocess.run(
        [sys.executable, str(Path(PROJECT_ROOT) / 'scripts' / 'build_model_meta.py')],
        check=True,
    )

df_base = pd.read_csv(CSV_PATH)
print(f'DataFrame cargado desde: {CSV_PATH.resolve()}')
print(f'Total imagenes: {len(df_base)}')
print()
print(df_base['class_name'].value_counts())


## 2. Parámetros de los extractores

In [ ]:
# Se muestran los parámetros configurados en `cad_tbx11k/features.py`.
# Modificar estos valores afecta directamente la dimensionalidad y el contenido
# del vector de características resultante.

print('Parámetros de los extractores de características:')
print()
print(f'  GLCM — propiedades calculadas  : {GLCM_PROPS}')
print(f'  GLCM — ángulos (rad)           : [0, π/4, π/2, 3π/4]')
print(f'  GLCM — niveles de gris         : 64 (img // 4)')
print()
print(f'  LBP — puntos de muestreo (P)   : {LBP_P}')
print(f'  LBP — radio (R)                : {LBP_R}')
print(f'  LBP — método                   : uniform')
print(f'  LBP — bins del histograma      : {LBP_P + 2}')
print()
print(f'  HOG — orientaciones            : 9')
print(f'  HOG — píxeles por celda        : {HOG_PIXELS_PER_CELL}')
print(f'  HOG — celdas por bloque        : (2, 2)')
print(f'  HOG — normalización de bloque  : L2-Hys')
print()
print(f'  Regiones anatómicas            : {list(REGIONS.keys())}')

## 3. Verificación de los extractores

In [ ]:
# Se verifica que cada extractor produce el número correcto de features
# sobre una imagen de muestra de cada clase antes de procesar el dataset completo.
# Esto detecta errores de configuración de forma temprana y económica.
# Se ejecutan assertions que fallan con mensaje claro si hay una discrepancia.

verify_extractors(df_base)

## 4. Extracción masiva de características

In [ ]:
# Se verifica que df_base tiene las columnas necesarias para la extracción.
# Si se cargó desde CSV, se reconstruyen label y class_name si faltan.

from pathlib import Path
from src.data import LABEL_MAP

if 'label' not in df_base.columns:
    df_base['label'] = df_base['class_name'].map(LABEL_MAP)

if 'class_name' not in df_base.columns:
    df_base['class_name'] = df_base['image_path'].apply(lambda p: Path(p).parent.name)
    df_base = df_base[df_base['class_name'].isin(LABEL_MAP)].reset_index(drop=True)
    df_base['label'] = df_base['class_name'].map(LABEL_MAP)

print(df_base.columns.tolist())
print(df_base['class_name'].value_counts())

In [ ]:
# Se extraen las características de TODAS las imágenes de df_base.
# El subconjunto lo controla N_MODELO en src/config.py:
#   None -> dataset completo (todas las imágenes; valor por defecto).
#   int  -> esa cantidad por clase (modo desarrollo, submuestreo).
# No se balancean las clases: el desbalanceo real del dataset (health/sick
# mayoritarias, tb minoritaria) se conserva y se aborda en el notebook 07.

if N_MODELO is None:
    features_df, errores = extract_all_features(
        df_base, sample=False, include_context=False, include_shape=False,
    )
else:
    features_df, errores = extract_all_features(
        df_base,
        sample=True,
        sample_size=N_MODELO,
        random_state=42,
        include_context=False,
        include_shape=False,
    )

if errores:
    print(f'\nImages with read errors ({len(errores)}):')
    for e in errores[:5]:
        print(f'  {e}')

## 5. Verificación de integridad del DataFrame de características

In [ ]:
# Se verifica que el DataFrame resultante está libre de valores nulos e infinitos,
# y que la distribución de clases es la esperada.
# Los valores nulos o infinitos en el vector de features pueden causar fallos
# silenciosos en los modelos de machine learning.

print_feature_summary(features_df)

In [ ]:
# Se muestra la tabla de features por grupo para confirmar la dimensionalidad
# esperada de cada extractor.

tabla_grupos = plot_feature_summary_table(features_df)
print('Resumen de características por grupo:')
print(tabla_grupos.to_string())

## 6. Visualización de características GLCM

In [ ]:
# Se visualizan las distribuciones de las cinco propiedades GLCM por clase.
# Si una propiedad presenta separación visual entre clases, tendrá mayor
# capacidad discriminativa en el modelo de clasificación.
# Referencia graycoprops: https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.graycoprops

plot_glcm_distributions(features_df)

## 7. Visualización del histograma LBP promedio

In [ ]:
# Se grafica el histograma LBP promedio por clase.
# Diferencias en la distribución de bins LBP indican que las texturas locales
# difieren entre clases, lo cual es útil para la clasificación.


plot_lbp_histograms(features_df)

## 8. Estadísticas descriptivas de las features globales

In [ ]:
# Se generan tablas descriptivas de las features de intensidad global por clase.
# Sirven como línea base para interpretar los resultados del modelo de clasificación.

grupos = get_feature_groups(features_df)
glob_cols = grupos['Intensidad global']

# --- Tabla 1: media por clase ---
tabla_mean = (features_df.groupby('class_name')[glob_cols].mean().round(4))
tabla_mean.index.name = 'Class'
print('Table 1 — Mean by class:')
print(tabla_mean.to_string())
print()

# --- Tabla 2: desviación estándar por clase ---
tabla_std = (features_df.groupby('class_name')[glob_cols].std().round(4))
tabla_std.index.name = 'Class'
print('Table 2 — Standard deviation by class:')
print(tabla_std.to_string())
print()

# --- Tabla 3: media ± std consolidada ---
tabla_resumen = tabla_mean.copy().astype(str)
for col in glob_cols:
    tabla_resumen[col] = tabla_mean[col].round(3).astype(str) + ' ± ' + tabla_std[col].round(3).astype(str)
tabla_resumen.index.name = 'Class'
print('Table 3 — Summary (mean ± std):')
print(tabla_resumen.to_string())

## 9. Exportación a CSV

In [ ]:
# Se reordena el DataFrame para que las columnas de metadatos queden primero
# y se exporta a CSV en la raíz del proyecto.
# Este archivo es la entrada para la siguiente etapa del pipeline (modelado ML).

meta_cols = ['image_path', 'class_name', 'label']
feat_cols = [c for c in features_df.columns if c not in meta_cols]

features_df = features_df[meta_cols + feat_cols]
features_df.to_csv(OUTPUT_CSV, index=False)

print(f'Guardado en : {OUTPUT_CSV}')
print(f'Shape       : {features_df.shape}')
print(f'Tamaño      : {os.path.getsize(OUTPUT_CSV) / 1024 / 1024:.2f} MB')
print()
print('Primeras 3 filas (columnas de metadatos + primeras 5 features):')
print(features_df[meta_cols + feat_cols[:5]].head(3).to_string(index=False))

## 10. Calidad de la máscara de segmentación

Resume área, balance, contraste y tasa de `fallback` por clase. Un `fallback`
alto indica que la máscara fina no superó el control de calidad y se usó el
campo torácico grueso.


In [ ]:
qc = features_df.groupby('class_name')[
    ['mask_area_fraction', 'mask_lr_balance', 'mask_contrast', 'mask_fallback']
].mean().round(3)
print('Tasa de fallback global:', round(float(features_df['mask_fallback'].mean()), 3))
qc


In [ ]:
# Manifiesto de features: columnas, grupos y vista usada por HOG.
import datetime
from src.features import QUALITY_COLS, get_feature_groups, model_feature_columns

model_columns = model_feature_columns(features_df)
manifest_features = {
    'n_images': int(len(features_df)),
    'n_features_model': int(len(model_columns)),
    'feature_columns': model_columns,
    'feature_groups': {k: len(v) for k, v in get_feature_groups(features_df).items()},
    'quality_columns': sorted(QUALITY_COLS),
    'hog_view': 'lung (exterior relleno con mediana pulmonar)',
    'segmentation': 'campo pulmonar con margen 20 px; prior vertical 2-82 %',
    'created_at': datetime.datetime.now().isoformat(timespec='seconds'),
}
with open(DATA_DIR / 'features_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_features, f, ensure_ascii=False, indent=2)
print(f"Manifiesto guardado: {len(model_columns)} features de modelo")


## Conclusiones

Los resultados indican que las métricas globales de intensidad capturan algunas diferencias entre clases, especialmente en el brillo promedio de las imágenes. Sin embargo, el elevado solapamiento observado en contraste y entropía sugiere que estas características son insuficientes para separar claramente las categorías. Por ello, resulta necesario complementar este análisis con descriptores de textura, capaces de capturar patrones espaciales más complejos presentes en las imágenes.

## Próximas etapas del proyecto

```
features_tbx11k.csv
       │
       ▼
Análisis de características (correlación, importancia)
       │
       ▼
Reducción de dimensionalidad (PCA, selección de features)
       │
       ▼
Entrenamiento de modelos de clasificación
       │
       ▼
Evaluación del desempeño (ROC, AUC, F1 por clase)
```
